# Future Planning – Scan-Worker (Colab / T4)

Dünner Starter für den GPU-Scan-Worker. **Ablauf:**

1. Repo `FP_APP` klonen (flach).
2. Scan-Worker + GPU-Stack installieren (eine Setup-Zelle).
3. Gradio-Worker mit `share=True` starten → Scan-Bundle (`video.*` + `poses.json`) hochladen.
4. Ergebnis: `scene.ply` (z-up, metrisch) + `layout.txt` (SpatialLM) für den Server-Adapter.

Der Geometrie-Kern läuft CPU-only; hier auf Colab kommen Depth Anything V2 Small + SpatialLM dazu.
Deploy-Idee (Worker + Space, Zeiger v0) steht im Brain: **POC-Demo-Architektur-HF**.

In [ ]:
# Zelle 2 – Repo flach klonen (privates Repo → GITHUB_PAT aus Colab-Secrets)
from google.colab import userdata
import os, subprocess

PAT = userdata.get("GITHUB_PAT")  # in Colab: 🔑 Secrets → GITHUB_PAT hinterlegen
URL = f"https://{PAT}@github.com/Bryan-HSLU/FP_APP"
if not os.path.isdir("FP_APP"):
    subprocess.run(["git", "clone", "--depth", "1", URL, "FP_APP"], check=True)
os.chdir("FP_APP")
print("cwd:", os.getcwd())

In [ ]:
# Zelle 3 – EINE Setup-Zelle: Worker + GPU-Stack installieren
# Basis (Worker-Kern + Colab-Bausteine, alle mit permissiver Lizenz):
!pip -q install -e services/scan-worker[worker] opencv-python-headless open3d torch transformers

# ---------------------------------------------------------------------------
# TODO(erster echter Colab-Lauf, Fahrplan Schritt 5): SpatialLM + TorchSparse.
# TorchSparse muss auf die CUDA-Version der Session kompiliert werden (langsam)
# → Drive-Wheel-Cache-Pattern, damit nur EINMAL kompiliert wird:
#
#   from google.colab import drive; drive.mount('/content/drive')
#   WHEEL_CACHE = '/content/drive/MyDrive/fp_wheels'
#   if <wheel im Cache vorhanden>:
#       pip install <cache-wheel>
#   else:
#       pip install <torchsparse-quelle>  # kompiliert
#       cp <gebautes-wheel> WHEEL_CACHE    # für nächste Session sichern
#   pip install spatiallm  # CC-BY-NC – nur Colab, nie feste Dependency!
#
# Exakte Pins/Reihenfolge werden beim ersten Lauf fixiert und hier eingetragen.
# ---------------------------------------------------------------------------

In [ ]:
# Zelle 4 – Worker starten (öffentliche share-URL)
from fp_scan_worker.worker import erstelle_app

app = erstelle_app()
app.launch(share=True)
# → Zeiger v0: die ausgegebene *.gradio.live-URL MANUELL als FP_SCAN_WORKER_URL
#   im Hugging-Face-Space hinterlegen. Gist-Automation folgt später.

## Colab-Stolperfallen

- **Grosse Videos** nicht direkt hochladen – über Google Drive einbinden (`drive.mount`), sonst reisst der Upload ab.
- **Session stirbt** (Timeout / Neustart) → Laufzeit ist weg: Setup-Zelle (Zelle 3) erneut ausführen. Mit Drive-Wheel-Cache bleibt das schnell.
- **share-URL wechselt** bei jedem `launch` → nach jedem Neustart `FP_SCAN_WORKER_URL` im Space aktualisieren (Zeiger v0).